# Concept Direction Experiment Harness

Unified notebook harness for concept-direction experimentation across Gemma 2 and Gemma 3 model variants.

This notebook is intended to be executed manually or via papermill. It keeps the original nine experimental phases, but it now manages resources more aggressively by opening fresh model sessions at phase boundaries and immediately offloading retained artifacts to CPU.

## Notebook Parameters

The next code cell is tagged for papermill and should remain the single source of flat experiment parameters.

In [ ]:
# Parameters - These will be injected by papermill during parameterized runs
EXPERIMENT_NAME = "gemma3_pt_capitals_states"
EXPERIMENT_CONFIG_NAME = "manual"

MODEL_FAMILY = "gemma3"
MODEL_VARIANT = "pt"
MODEL_NAME_OVERRIDE = None
TRANSCODER_SET_OVERRIDE = None

CONCEPT_PAIR_NAME = "capitals_states"
PROMPT_OVERRIDE = None
PROMPT_RENDER_MODE = "plain"

TOP_N = 10
DEFAULT_SCALE_FACTOR = 10.0
SCALE_FACTOR_SWEEP = [2.0, 5.0, 10.0, 20.0, 50.0]
ABLATION_N_LIST = [5, 10, 25, 50, 100]
ENABLE_SIGN_AWARE = True

# Optional key token override — list of tokens to track in sanity check & displays.
# When None, uses concept_pair.key_tokens from the experimentation module.
KEY_TOKENS = None

# TARGET_TOKENS takes precedence over TARGET_TOKEN_IDS when both are provided.
TARGET_TOKENS = None
TARGET_TOKEN_IDS = (24278, 26057)

NEURONPEDIA_MODEL_OVERRIDE = None
NEURONPEDIA_SET_OVERRIDE = None
FORCE_DEVICE = None
EXPERIMENT_WORK_DIR = None
VERBOSE = True

# Resource overrides — set via YAML configs to reduce peak VRAM for large/IT models.
# When None, cfg_aliases defaults apply (batch_size=256, max_feature_nodes=8192).
BATCH_SIZE = None
MAX_FEATURE_NODES = None

# V12 approach controls — which extraction mode and intervention type to use.
# "answer_position_state" = default (approaches a, b, c)
# "context_enhanced" = context-enhanced extraction (approaches d, e)
STORE_LATENT_EXTRACTION_MODE = "answer_position_state"
CONTEXT_ENHANCED_SCALE = 1.0

In [ ]:
from __future__ import annotations

import gc
import json
import os
import shutil
from typing import Any

import matplotlib.pyplot as plt
import torch

# Reduce CUDA memory fragmentation — recommended by PyTorch OOM diagnostics.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

from it_examples.utils.nb_ui_utils import (
    display_ablation_chart,
    display_key_token_logits,
    display_top_features_comparison,
    display_topk_token_predictions,
)
from tests.concept_direction_approach_parity.concept_direction_approach_experimentation import (
    CONCEPT_PAIRS,
    ConceptPair,
    resolve_intervention_prompt,
)
from tests.concept_direction_approach_parity.concept_direction_experiment_utils import (
    NotebookHarnessConfig,
    collect_feature_pool as _collect_feature_pool,
    compute_embed_direction as _compute_embed_direction,
    compute_store_direction as _compute_store_direction,
    phase_run_name as _phase_run_name,
    run_ablations as _run_ablations,
    run_direction_probes as _run_direction_probes,
    run_pipeline as _run_pipeline,
    run_initial_sanity_check as _run_initial_sanity_check,
    run_scale_sweep as _run_scale_sweep,
    run_direct_projection_pipeline as _run_direct_projection_pipeline,
    run_direct_projection_scale_sweep as _run_direct_projection_scale_sweep,
    run_sign_aware as _run_sign_aware,
    run_tokenizer_verification as _run_tokenizer_verification,
)
from tests.concept_direction_approach_parity.experiment_resource_utils import (
    create_work_root,
    experiment_session,
    feature_ids_to_tuples,
    resolve_model_spec,
    tensor_to_cpu,
)

print("Imports complete.")

In [ ]:
MODEL_SPEC = resolve_model_spec(
    MODEL_FAMILY,
    MODEL_VARIANT,
    model_name_override=MODEL_NAME_OVERRIDE,
    transcoder_set_override=TRANSCODER_SET_OVERRIDE,
    neuronpedia_model_override=NEURONPEDIA_MODEL_OVERRIDE,
    neuronpedia_set_override=NEURONPEDIA_SET_OVERRIDE,
)

MODEL_NAME = MODEL_SPEC.model_name
TRANSCODER_SET = MODEL_SPEC.transcoder_set
HF_MODEL_HEAD = MODEL_SPEC.hf_model_head
NEURONPEDIA_MODEL = MODEL_SPEC.neuronpedia_model
NEURONPEDIA_SET = MODEL_SPEC.neuronpedia_set
WORK_ROOT = create_work_root(EXPERIMENT_WORK_DIR, EXPERIMENT_NAME)

concept_pair: ConceptPair = CONCEPT_PAIRS[CONCEPT_PAIR_NAME]
prompt = PROMPT_OVERRIDE or resolve_intervention_prompt(
    concept_pair,
    use_chat_template=PROMPT_RENDER_MODE != "plain",
)

NOTEBOOK_CFG = NotebookHarnessConfig(
    experiment_name=EXPERIMENT_NAME,
    experiment_config_name=EXPERIMENT_CONFIG_NAME,
    model_family=MODEL_FAMILY,
    model_variant=MODEL_VARIANT,
    model_name=MODEL_NAME,
    transcoder_set=TRANSCODER_SET,
    hf_model_head=HF_MODEL_HEAD,
    neuronpedia_model=NEURONPEDIA_MODEL,
    neuronpedia_set=NEURONPEDIA_SET,
    concept_pair_name=CONCEPT_PAIR_NAME,
    prompt=prompt,
    prompt_render_mode=PROMPT_RENDER_MODE,
    target_tokens=tuple(TARGET_TOKENS) if TARGET_TOKENS is not None else None,
    target_token_ids=tuple(TARGET_TOKEN_IDS) if TARGET_TOKEN_IDS is not None else None,
    top_n=TOP_N,
    default_scale_factor=DEFAULT_SCALE_FACTOR,
    scale_factor_sweep=SCALE_FACTOR_SWEEP,
    ablation_n_list=ABLATION_N_LIST,
    enable_sign_aware=ENABLE_SIGN_AWARE,
    force_device=FORCE_DEVICE,
    work_root=WORK_ROOT,
    batch_size=BATCH_SIZE,
    max_feature_nodes=MAX_FEATURE_NODES,
    key_tokens_override=tuple(KEY_TOKENS) if KEY_TOKENS is not None else None,
    store_latent_extraction_mode=STORE_LATENT_EXTRACTION_MODE,
    context_enhanced_scale=CONTEXT_ENHANCED_SCALE,
)

print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Config name: {EXPERIMENT_CONFIG_NAME}")
print(f"Model: {MODEL_NAME} ({MODEL_FAMILY}:{MODEL_VARIANT})")
print(f"Transcoder set: {TRANSCODER_SET}")
print(f"HF model head override: {HF_MODEL_HEAD}")
print(f"Concept pair: {concept_pair.name}")
print(f"Resolved prompt: {NOTEBOOK_CFG.prompt}")
print(f"Prompt render mode: {NOTEBOOK_CFG.prompt_render_mode}")
print(f"Configured target tokens: {NOTEBOOK_CFG.target_tokens}")
print(f"Configured target token ids: {NOTEBOOK_CFG.target_token_ids}")
print(f"Batch size override: {BATCH_SIZE}")
print(f"Max feature nodes override: {MAX_FEATURE_NODES}")
print(f"Work root: {WORK_ROOT}")

In [ ]:
def phase_run_name(label: str) -> str:
    return _phase_run_name(NOTEBOOK_CFG, label)


def run_tokenizer_verification() -> dict[str, Any]:
    return _run_tokenizer_verification(NOTEBOOK_CFG)


def run_initial_sanity_check() -> dict[str, Any]:
    return _run_initial_sanity_check(NOTEBOOK_CFG)


def compute_embed_direction() -> dict[str, Any]:
    return _compute_embed_direction(NOTEBOOK_CFG)


def run_pipeline(
    direction: torch.Tensor,
    label: str,
    *,
    scale_factor: float,
    top_n: int,
    group_a_ids: list[int],
    group_b_ids: list[int],
) -> dict[str, Any]:
    return _run_pipeline(
        NOTEBOOK_CFG,
        direction,
        label,
        scale_factor=scale_factor,
        top_n=top_n,
        group_a_ids=group_a_ids,
        group_b_ids=group_b_ids,
    )


def compute_store_direction() -> dict[str, Any]:
    return _compute_store_direction(NOTEBOOK_CFG)


def run_scale_sweep(direction: torch.Tensor, group_a_ids: list[int], group_b_ids: list[int]) -> list[dict[str, Any]]:
    return _run_scale_sweep(NOTEBOOK_CFG, direction, group_a_ids, group_b_ids)


def collect_feature_pool(
    direction: torch.Tensor,
    group_a_ids: list[int],
    group_b_ids: list[int],
    *,
    top_n: int,
) -> dict[str, Any]:
    return _collect_feature_pool(NOTEBOOK_CFG, direction, group_a_ids, group_b_ids, top_n=top_n)


def run_ablations(
    feature_pool: dict[str, Any],
    pre_logits_ref: torch.Tensor,
) -> tuple[dict[str, dict[str, float]], dict[str, float], dict[int, torch.Tensor]]:
    return _run_ablations(NOTEBOOK_CFG, feature_pool, pre_logits_ref)


def run_sign_aware(feature_pool: dict[str, Any], pre_logits_ref: torch.Tensor) -> dict[str, Any]:
    return _run_sign_aware(NOTEBOOK_CFG, feature_pool, pre_logits_ref)


def run_direct_projection_pipeline(
    direction: torch.Tensor,
    label: str,
    *,
    scale_factor: float,
) -> dict[str, Any]:
    return _run_direct_projection_pipeline(NOTEBOOK_CFG, direction, label, scale_factor=scale_factor)


def run_direct_projection_scale_sweep(direction: torch.Tensor) -> list[dict[str, Any]]:
    return _run_direct_projection_scale_sweep(NOTEBOOK_CFG, direction)


def run_direction_probes(embed_direction: torch.Tensor, store_direction: torch.Tensor) -> dict[str, Any]:
    return _run_direction_probes(NOTEBOOK_CFG, embed_direction, store_direction)


RESULTS: dict[str, Any] = {}

## Phase 1: Session Initialization and Tokenizer Verification

In [ ]:
tokenizer_report = run_tokenizer_verification()
RESULTS["tokenizer_report"] = tokenizer_report

print(f"Module type: {tokenizer_report['module_type']}")
print(f"Prompt render mode: {tokenizer_report['prompt_render_mode']}")
print(f"Prompt token count: {tokenizer_report['prompt_token_count']}")
print(
    "Target tokens: "
    f"A={tokenizer_report['target_tokens']['group_a']['id']} ({tokenizer_report['target_tokens']['group_a']['decoded']!r}), "
    f"B={tokenizer_report['target_tokens']['group_b']['id']} ({tokenizer_report['target_tokens']['group_b']['decoded']!r})"
)

for group_name, entries in tokenizer_report["groups"].items():
    print(f"\n{group_name}:")
    for entry in entries:
        print(f"  {entry['token']}: ids={entry['ids']}, decoded='{entry['decoded']}'")

print("\nKey tokens:")
for token, payload in tokenizer_report["key_tokens"].items():
    print(f"  {token}: ids={payload['ids']}, decoded='{payload['decoded']}'")

print("\nRendered prompt variants:")
for mode_name, preview in tokenizer_report["render_variants"].items():
    if preview is None:
        print(f"\n[{mode_name}]\n(not available — model has no chat template)")
        continue
    print(f"\n[{mode_name}]\n{preview[:300]}")
    print(f"token_ids={tokenizer_report['render_variant_token_ids'][mode_name]}")
    print(f"tokens={tokenizer_report['render_variant_tokens'][mode_name][:40]}")

equalities = tokenizer_report["render_variant_equalities"]
print("\nPrompt parity checks:")
print(f"  apply_chat_template vs gemma_dataclass string parity: {equalities['apply_chat_template_vs_dataclass']}")
print(
    "  apply_chat_template vs gemma_dataclass token-id parity: "
    f"{equalities['apply_chat_template_vs_dataclass_token_ids']}"
)

print("\nSelected prompt preview:")
print(tokenizer_report["selected_prompt_preview"])
print(f"Selected prompt token ids: {tokenizer_report['selected_prompt_token_ids']}")
print(f"Selected prompt tokens: {tokenizer_report['selected_prompt_tokens'][:60]}")

In [ ]:
from it_examples.utils.nb_ui_utils import format_prob as _fp

reasoning_check = run_initial_sanity_check()
RESULTS["initial_sanity_check"] = reasoning_check

print("Initial Sanity Check")
print(f"Prompt style: {reasoning_check['prompt_style']}")
print(f"Prompt: {reasoning_check['rendered_prompt'][:200]}")
print(f"Generated: {reasoning_check['generated_text']}")

print("\nFirst-token logit analysis:")
print(f"  {'Token':<12} {'ID':>8} {'Logit':>10} {'Prob':>12}")
print(f"  {'-' * 46}")
for entry in reasoning_check["key_tokens"]:
    if entry.get("token_id") is not None:
        print(f"  {entry['label']:<12} {entry['token_id']:>8} {entry['logit']:>10.4f} {_fp(entry['prob']):>12}")
    else:
        print(f"  {entry['label']:<12} {'N/A':>8} {'N/A':>10} {'N/A':>12}")
print(f"  {'-' * 46}")
print(
    f"  {'Top-1':<12} {reasoning_check['top1_id']:>8} {'':>10} {_fp(reasoning_check['top1_prob']):>12}  ({reasoning_check['top1_token']!r})"
)

## Phase 2 and 3: Embed-Based Direction and Full Pipeline

In [ ]:
embed_direction_result = compute_embed_direction()
embed_pipeline = run_pipeline(
    embed_direction_result["direction"],
    "Embed",
    scale_factor=DEFAULT_SCALE_FACTOR,
    top_n=TOP_N,
    group_a_ids=embed_direction_result["group_a_ids"],
    group_b_ids=embed_direction_result["group_b_ids"],
)
RESULTS["embed_direction_result"] = embed_direction_result
RESULTS["embed_pipeline"] = embed_pipeline
print(f"Embed direction norm: {torch.linalg.vector_norm(embed_direction_result['direction']):.6f}")
print(f"Pre-gap ({embed_pipeline['target_a_tok']}−{embed_pipeline['target_b_tok']}): {embed_pipeline['pre_gap']:.4f}")
print(f"Post-gap: {embed_pipeline['post_gap']:.4f}")
print(f"Gap Δ: {embed_pipeline['gap_delta']:+.4f}")
display_top_features_comparison(
    {"Embed — Top Features": embed_pipeline["feature_ids"]},
    {"Embed — Top Features": embed_pipeline["feature_scores"]},
    neuronpedia_model=NEURONPEDIA_MODEL,
    neuronpedia_set=NEURONPEDIA_SET,
)

In [ ]:
with experiment_session(
    WORK_ROOT,
    phase_run_name("embed_display"),
    model_family=MODEL_FAMILY,
    model_name=MODEL_NAME,
    transcoder_set=TRANSCODER_SET,
    force_device=FORCE_DEVICE,
) as (_, _, tokenizer):
    display_topk_token_predictions(
        prompt,
        embed_pipeline["pre_logits"],
        embed_pipeline["post_logits"],
        tokenizer,
        k=5,
        key_tokens=[
            (label, token_id) for label, token_id in zip(embed_pipeline["key_labels"], embed_pipeline["key_ids"])
        ],
    )
    display_key_token_logits(
        embed_pipeline["pre_logits"],
        embed_pipeline["post_logits"],
        embed_pipeline["key_ids"],
        embed_pipeline["key_labels"],
        title=f"Embed {DEFAULT_SCALE_FACTOR}× — Key-Token Logits",
    )

## Phase 4 and 5: Store-Based Direction and Summary

In [ ]:
store_direction_result = compute_store_direction()
store_pipeline = run_pipeline(
    store_direction_result["direction"],
    "Store",
    scale_factor=DEFAULT_SCALE_FACTOR,
    top_n=TOP_N,
    group_a_ids=store_direction_result["group_a_ids"],
    group_b_ids=store_direction_result["group_b_ids"],
)
RESULTS["store_direction_result"] = store_direction_result
RESULTS["store_pipeline"] = store_pipeline
cosine_similarity = float(
    torch.nn.functional.cosine_similarity(
        embed_direction_result["direction"].unsqueeze(0), store_direction_result["direction"].unsqueeze(0)
    ).item()
)
embed_set = set(embed_pipeline["feature_ids"])
store_set = set(store_pipeline["feature_ids"])
shared = embed_set & store_set
total = embed_set | store_set
feature_jaccard = len(shared) / len(total) if total else 0.0
RESULTS["comparison"] = {
    "cosine_similarity": cosine_similarity,
    "feature_jaccard": feature_jaccard,
    "shared_feature_count": len(shared),
    "total_feature_count": len(total),
}
print(f"Store direction norm: {torch.linalg.vector_norm(store_direction_result['direction']):.6f}")
print(f"Predictions: {store_direction_result['prediction_info']['n_correct']}/{store_direction_result['n_total']}")
print(f"Cosine similarity: {cosine_similarity:.6f}")
print(f"Feature Jaccard: {feature_jaccard:.4f} ({len(shared)}/{len(total)})")
print(f"Embed Δ: {embed_pipeline['gap_delta']:+.4f} | Store Δ: {store_pipeline['gap_delta']:+.4f}")

In [ ]:
with experiment_session(
    WORK_ROOT,
    phase_run_name("store_display"),
    model_family=MODEL_FAMILY,
    model_name=MODEL_NAME,
    transcoder_set=TRANSCODER_SET,
    force_device=FORCE_DEVICE,
) as (_, _, tokenizer):
    display_top_features_comparison(
        {
            "Embed — Top Features": embed_pipeline["feature_ids"],
            "Store — Top Features": store_pipeline["feature_ids"],
        },
        {
            "Embed — Top Features": embed_pipeline["feature_scores"],
            "Store — Top Features": store_pipeline["feature_scores"],
        },
        neuronpedia_model=NEURONPEDIA_MODEL,
        neuronpedia_set=NEURONPEDIA_SET,
    )
    display_topk_token_predictions(
        prompt,
        store_pipeline["pre_logits"],
        store_pipeline["post_logits"],
        tokenizer,
        k=5,
        key_tokens=[
            (label, token_id) for label, token_id in zip(store_pipeline["key_labels"], store_pipeline["key_ids"])
        ],
    )
    display_key_token_logits(
        store_pipeline["pre_logits"],
        store_pipeline["post_logits"],
        store_pipeline["key_ids"],
        store_pipeline["key_labels"],
        title=f"Store {DEFAULT_SCALE_FACTOR}× — Key-Token Logits",
    )

## Phase 4b: Direct Projection (Store Direction)

Bypasses feature selection entirely — adds the scaled store concept direction
vector directly to the residual stream at `unembed.hook_in` (lm_head input).

In [ ]:
dp_pipeline = run_direct_projection_pipeline(
    store_direction_result["direction"],
    "DirectProj_Store",
    scale_factor=DEFAULT_SCALE_FACTOR,
)
RESULTS["dp_pipeline"] = dp_pipeline
print(f"Direct Projection (Store, {DEFAULT_SCALE_FACTOR}×):")
print(f"  Pre gap:  {dp_pipeline['pre_gap']:+.4f}")
print(f"  Post gap: {dp_pipeline['post_gap']:+.4f}")
print(f"  Δ gap:    {dp_pipeline['gap_delta']:+.4f}")
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

In [ ]:
with experiment_session(
    WORK_ROOT,
    phase_run_name("dp_display"),
    model_family=MODEL_FAMILY,
    model_name=MODEL_NAME,
    transcoder_set=TRANSCODER_SET,
    force_device=FORCE_DEVICE,
) as (_, _, tokenizer):
    display_topk_token_predictions(
        prompt,
        dp_pipeline["pre_logits"],
        dp_pipeline["post_logits"],
        tokenizer,
        k=5,
        key_tokens=[(label, token_id) for label, token_id in zip(dp_pipeline["key_labels"], dp_pipeline["key_ids"])],
    )
    display_key_token_logits(
        dp_pipeline["pre_logits"],
        dp_pipeline["post_logits"],
        dp_pipeline["key_ids"],
        dp_pipeline["key_labels"],
        title=f"Direct Projection (Store) {DEFAULT_SCALE_FACTOR}× — Key-Token Logits",
    )

## Phase 6: Scale Factor Sweep

In [ ]:
sweep_results = run_scale_sweep(
    embed_direction_result["direction"],
    embed_direction_result["group_a_ids"],
    embed_direction_result["group_b_ids"],
)
RESULTS["sweep_results"] = sweep_results
for sweep_result in sweep_results:
    print(f"Scale={sweep_result['scale_factor']:.1f}x Δ={sweep_result['gap_delta']:+.4f}")
scales = [entry["scale_factor"] for entry in sweep_results]
deltas = [entry["gap_delta"] for entry in sweep_results]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(scales, deltas, "o-", color="#2471A3", linewidth=2, markersize=8)
ax.axhline(y=0, color="gray", linestyle="--", alpha=0.5)
ax.set_xlabel("Scale Factor")
ax.set_ylabel(f"Gap Δ ({embed_pipeline['target_a_tok']}−{embed_pipeline['target_b_tok']})")
ax.set_title(f"Scale Factor Sweep — {MODEL_FAMILY}:{MODEL_VARIANT} {concept_pair.name}")
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()
plt.close(fig)
for sweep_result in [sweep_results[0], sweep_results[-1]]:
    display_key_token_logits(
        sweep_result["pre_logits"],
        sweep_result["post_logits"],
        sweep_result["key_ids"],
        sweep_result["key_labels"],
        title=f"Embed {sweep_result['scale_factor']}× — Key-Token Logits",
    )

## Phase 7 and 8: Progressive Ablation and Sign-Aware Feature Selection

In [ ]:
feature_pool = collect_feature_pool(
    embed_direction_result["direction"],
    embed_direction_result["group_a_ids"],
    embed_direction_result["group_b_ids"],
    top_n=max(max(ABLATION_N_LIST), TOP_N * 2),
)

RESULTS["feature_pool_summary"] = {
    "feature_count": int(feature_pool["feature_ids"].shape[0]),
    "target_a_id": feature_pool["target_a_id"],
    "target_b_id": feature_pool["target_b_id"],
}

ablation_groups, ablation_logit_diffs, ablation_results = run_ablations(feature_pool, embed_pipeline["pre_logits"])
RESULTS["ablation_logit_diffs"] = ablation_logit_diffs

display_ablation_chart(
    ablation_groups,
    logit_diffs=ablation_logit_diffs,
    title=f"Embed ablation — {MODEL_FAMILY}:{MODEL_VARIANT} {concept_pair.name}",
)

if ablation_results:
    strongest_n = max(ablation_results)
    display_key_token_logits(
        embed_pipeline["pre_logits"],
        ablation_results[strongest_n],
        feature_pool["key_ids"],
        feature_pool["key_labels"],
        title=f"Top-{strongest_n} Ablation — Key-Token Logits",
    )

if ENABLE_SIGN_AWARE:
    sign_aware = run_sign_aware(feature_pool, embed_pipeline["pre_logits"])
    RESULTS["sign_aware"] = {
        "positive_feature_count": int(len(sign_aware["positive_features"])),
        "negative_feature_count": int(len(sign_aware["negative_features"])),
        "messages": list(sign_aware.get("messages", [])),
    }
    for message in sign_aware.get("messages", []):
        print(f"\n⚠️  Sign-aware: {message}")
    has_pos = len(sign_aware["positive_features"]) > 0
    has_neg = len(sign_aware["negative_features"]) > 0
    if has_pos or has_neg:
        feature_display = {}
        score_display = {}
        if has_pos:
            feature_display["Positive activation"] = feature_ids_to_tuples(sign_aware["positive_features"][:TOP_N])
            score_display["Positive activation"] = tensor_to_cpu(sign_aware["positive_scores"][:TOP_N]).tolist()
        if has_neg:
            feature_display["Negative activation"] = feature_ids_to_tuples(sign_aware["negative_features"][:TOP_N])
            score_display["Negative activation"] = tensor_to_cpu(sign_aware["negative_scores"][:TOP_N]).tolist()
        display_top_features_comparison(
            feature_display,
            score_display,
            neuronpedia_model=NEURONPEDIA_MODEL,
            neuronpedia_set=NEURONPEDIA_SET,
        )
    else:
        print("\n⚠️  Sign-aware: No features had non-zero activations — nothing to display.")
    if "positive_post_logits" in sign_aware:
        display_key_token_logits(
            embed_pipeline["pre_logits"],
            sign_aware["positive_post_logits"],
            feature_pool["key_ids"],
            feature_pool["key_labels"],
            title=f"Positive features {DEFAULT_SCALE_FACTOR}× — Key-Token Logits",
        )
    if "negative_post_logits" in sign_aware:
        display_key_token_logits(
            embed_pipeline["pre_logits"],
            sign_aware["negative_post_logits"],
            feature_pool["key_ids"],
            feature_pool["key_labels"],
            title="Negative features ablated — Key-Token Logits",
        )
else:
    print("\nSign-aware feature analysis: DISABLED (set ENABLE_SIGN_AWARE=true to enable)")

## Phase 9: Direction Consistency Probes

In [ ]:
probe_results = run_direction_probes(embed_direction_result["direction"], store_direction_result["direction"])
RESULTS["probe_results"] = probe_results
for label, payload in probe_results.items():
    print(f"\n{label} direction probes:")
    print(f"{'Token':<15} {'Projection':>12} {'Group':>8}")
    print(f"{'-' * 15} {'-' * 12} {'-' * 8}")
    for row in payload["rows"]:
        print(f"{row['token']:<15} {row['projection']:>12.4f} {row['group']:>8}")
    separation = payload["mean_a"] - payload["mean_b"]
    print(f"Mean A: {payload['mean_a']:.4f}, Mean B: {payload['mean_b']:.4f}, Separation: {separation:.4f}")

## Cleanup

In [ ]:
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

if EXPERIMENT_WORK_DIR is None:
    shutil.rmtree(WORK_ROOT, ignore_errors=True)

SUMMARY_RECORD = {
    "experiment_name": EXPERIMENT_NAME,
    "config_name": EXPERIMENT_CONFIG_NAME,
    "model_family": MODEL_FAMILY,
    "model_variant": MODEL_VARIANT,
    "model_name": MODEL_NAME,
    "concept_pair": concept_pair.name,
    "prompt_render_mode": NOTEBOOK_CFG.prompt_render_mode,
    "target_tokens": list(NOTEBOOK_CFG.target_tokens) if NOTEBOOK_CFG.target_tokens is not None else None,
    "target_token_ids": list(NOTEBOOK_CFG.target_token_ids) if NOTEBOOK_CFG.target_token_ids is not None else None,
    "embed_gap_delta": RESULTS["embed_pipeline"]["gap_delta"],
    "store_gap_delta": RESULTS["store_pipeline"]["gap_delta"],
    "cosine_similarity": RESULTS["comparison"]["cosine_similarity"],
    "feature_jaccard": RESULTS["comparison"]["feature_jaccard"],
    "prediction_correct": RESULTS["store_direction_result"]["prediction_info"]["n_correct"],
    "prediction_total": RESULTS["store_direction_result"]["n_total"],
    "work_root_removed": EXPERIMENT_WORK_DIR is None,
}

print("Notebook complete.")
print(json.dumps(SUMMARY_RECORD, indent=2))